# TV posterior-mean example — recover a denoiser's implicit regularizer

Recover `f_reg`, the implicit regularizer of the anisotropic-TV posterior-mean
denoiser, from **denoiser evaluations alone**, with **one convex network**
(`work2.tex` Instantiation B).

All mathematics lives in the `tvpm/` package.

**Pipeline:** data (`u_PM` from the sampler) → fit one convex net → held-out prox
residual → what the prior looks like → **the prior in action**: denoise an image
with the learned prior and compare against the true posterior-mean denoiser.
Method and reproduction: `README.md`. Historical record: `DESIGN.md`. Numbers: `results/`.

## Configuration

The choices are grouped by the **stage** they act on. The two stages share
nothing but the cached array between them: `PM_SWEEPS` changes the data and the
noise floor, `FIT_STEPS` changes only how well the net fits whatever data it is
given.

Every output is keyed on `(SIGMA, T)` — and on `PM_SWEEPS`, `FIT_STEPS`, `ARCH`
and `BETA` — so changing any of them writes to new files and **cannot overwrite
an earlier run's data, checkpoints, or figures**.


In [ ]:
# Auto-reload the tvpm package when its files change on disk, so edits to
# tvpm/*.py take effect without a kernel restart.
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.getcwd())            # tv_pm/, which holds tvpm/

import numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

from tvpm import dataset, denoise, figures, recover
from tvpm.sampler import from_sigma_t

# =========================================================================
# STAGE 1 -- THE DATA: u_PM(x_k), from the MCMC sampler
# =========================================================================
# The model's two degrees of freedom.
# SIGMA is the standard deviation of the Gaussian noise contaminating the image.
# T is the denoising strength (as EPS -> 0, u_PM -> the MAP, prox_{T*ATV}).
# EPS is the temperature -- i.e., exactly how much f_reg is a SMOOTHED TV rather
# than TV. (The MATLAB's tabulated pair is sigma=10/256, t=16/256; the pair
# below is this experiment's choice.)
SIGMA = 20 / 256
T     = 20 / 256
EPS, LAM = from_sigma_t(SIGMA, T)

# MCMC chain length per patch -- "m" in README.md and DESIGN.md. One sweep
# updates every pixel once; the first PM_SWEEPS//5 are burn-in and the rest are
# averaged into u_PM. Raising it lowers `delta`, the sampler noise floor that
# the prox residual is read against.
# Must be >= 25. Suggested: 8000 for full run.
#
PM_SWEEPS = 8000

# Fraction of the full patch counts (20000 train / 4000 val / 4000 eval).
# Below 1.0 the splits are smaller and cheaper, and land in their own files.
SCALE = 1.0

# =========================================================================
# STAGE 2 -- THE FIT: one convex net trained on those u_PM
# =========================================================================
ARCH      = "fc"    # "fc" = dense ICNN (the LPN);    "conv" = convolutional ICNN
BETA      = 20      # Softplus sharpness, matched to the TV kink
FIT_STEPS = 250000  # Adam steps for training the network. Suggested: 250000 for full run.

REFIT     = False   # True: retrain even if a checkpoint for this exact
                    # config exists (overwrites it). Changing any config
                    # value above already forces a fresh fit on its own.

# =========================================================================
# THE EXAMPLE IMAGE
# =========================================================================
# Image denoising with the recovered prior.
# train and val patches come from Barbara and the eval split from cameraman,
# both fixed in dataset.SPLITS and unaffected by this setting. It defaults to
# cameraman so that section 4 and the scores in section 3 concern the same image.
IMAGE = "cameraman_256x256_d"

TAG = dataset.tag(SIGMA, T)
print(f"stage 1 (data): sigma={SIGMA:.5f}  t={T:.5f}   ->  eps={EPS:.5f}  lam={LAM:.5f}")
print(f"                m={PM_SWEEPS}  scale={SCALE:g}"
      f"   [{'default (sigma,t)' if not TAG else TAG}]")
print(f"stage 2 (fit) : arch={ARCH}  beta={BETA}  steps={FIT_STEPS}")
print(f"example image : {IMAGE}   (train=barbara, held-out eval=cameraman)")

## 0. What this run will compute


In [ ]:
# Step 0 -- what THIS run will actually do, given the config above (REFIT included).
todo = dataset.plan(PM_SWEEPS, SIGMA, T, SCALE)
cached = recover.find_checkpoint(arch=ARCH, sweeps=PM_SWEEPS, beta=BETA,
                                 steps=FIT_STEPS, sigma=SIGMA, t=T)
will_fit = REFIT or cached is None            # REFIT overrides the cache

print("stage 1  data     :", f"sample {todo}  (~15 min at PM_SWEEPS=8000, full scale)"
      if todo else "cached, nothing to do")
if not will_fit:
    print("stage 2  network  : cached, nothing to do")
else:
    why = "REFIT=True, retrain over the cached checkpoint" if cached is not None \
          else "no checkpoint for this config, train from scratch"
    print(f"stage 2  network  : {why}  (~2 h fc / ~13 h conv at FIT_STEPS=250000)")

# Sections 3-5 ALWAYS run the sampler, cache or not: estimate_delta measures the
# noise floor, and the denoise/cost cells resample u_PM for the comparison.
print("sections 3-5      : always sample (delta, denoise, cost) -- by design")

if todo or will_fit:
    print("\nRunning the cells below will do the above. Reduce PM_SWEEPS, FIT_STEPS,")
    print("SCALE, or set REFIT=False if that is more than you meant to spend.")


## 1. Data — the sampler's `u_PM(x_k)`

MCMC over 8×8 patches. `ensure` samples only what is missing and caches it in `data/`.

Train and val are drawn from Barbara at **distinct patch positions**, so they are
rebuilt together — the position bookkeeping that keeps them disjoint cannot be
reconstructed from a cached file. `eval` is cameraman, with its own position pool.


In [ ]:
tr, va, ev = dataset.ensure(sweeps=PM_SWEEPS, sigma=SIGMA, t=T, scale=SCALE)
print("patches:", {k: tuple(v.shape) for k, v in
                   [("train", tr["y"]), ("val", va["y"]), ("eval", ev["y"])]})


## 2. The fit — one convex net on those `u_PM`

`ensure_model` loads the checkpoint matching this configuration, or trains one
and saves it under a name carrying every hyperparameter. Either way the rest of
the notebook is identical, and a second run of this cell is instant.


In [ ]:
model, units, ck = recover.ensure_model(
    arch=ARCH, sweeps=PM_SWEEPS, beta=BETA, steps=FIT_STEPS,
    sigma=SIGMA, t=T, scale=SCALE, standardize=True, force=REFIT)

ckpt_path = recover.find_checkpoint(arch=ARCH, sweeps=PM_SWEEPS, beta=BETA,
                                    steps=FIT_STEPS, sigma=SIGMA, t=T)
print(f"{ARCH}-ICNN, {sum(p.numel() for p in model.parameters())} parameters")
print(f"checkpoint: {os.path.basename(ckpt_path)}")


### Held-out prox residual

The recovery score, with no ground truth: `‖∇J_θ(y) − (x−y)‖` on held-out
patches, relative to the target `x−y`. Read it against the sampler noise floor
`δ` — the residual cannot be expected to fall below the noise in its own targets.
`eval` is the transfer image (cameraman); `val` is the training distribution
(Barbara).


In [ ]:
delta = recover.estimate_delta(ev["x"], PM_SWEEPS, sigma=SIGMA, t=T)
s_ev = recover.score(model, ev, units, delta)
s_va = recover.score(model, va, units, delta)

print(f"sampler noise floor delta          : {100*delta:.2f} %")
print(f"prox residual  eval (cameraman)    : {100*s_ev['resid_rel']:.2f} %  "
      f"= {s_ev['ratio']:.1f} x delta   |  corr(J,TV) {s_ev['tv_corr']:+.3f}")
print(f"prox residual  val  (barbara)      : {100*s_va['resid_rel']:.2f} %  "
      f"= {s_va['ratio']:.1f} x delta   |  corr(J,TV) {s_va['tv_corr']:+.3f}")


## 3. What the recovered prior looks like

`J_θ` vs TV (correlated, not equal — a *smoothed* TV, which is what the theory
predicts at `EPS > 0`); the prior penalising structure.
For the conv net, the learned first-layer kernels — many emerge as
`[+1,−1]` difference stencils, TV's building blocks.


In [ ]:
fig, corr = figures.figure_core(model, units.mu, units.s,
                                {"arch": ARCH, "beta": BETA,
                                 "sweeps": PM_SWEEPS, "steps": FIT_STEPS}, ev)
p = os.path.join(figures.OUT, f"step4_core_{ARCH}{TAG}.png")
os.makedirs(figures.OUT, exist_ok=True)
fig.savefig(p, dpi=150, bbox_inches="tight"); plt.close(fig)
print(f"corr(J, TV) = {corr:.3f}   ->  {os.path.basename(p)}")
display(Image(filename=p))


In [ ]:
if ARCH == "conv":
    fk = figures.figure_kernels(model, {"arch": ARCH,
                                   "sweeps": PM_SWEEPS, "steps": FIT_STEPS})
    pk = os.path.join(figures.OUT, f"step4_kernels_{ARCH}{TAG}.png")
    fk.savefig(pk, dpi=150, bbox_inches="tight"); plt.close(fk)
    display(Image(filename=pk))
else:
    print("the learned-kernel figure is conv-only (set ARCH='conv')")


## 4. Denoising with the prior and compared to the true denoiser

The posterior-mean denoiser is the proximal operator of `f_reg`, so if the
recovery is faithful the prox of the learned prior must reproduce it:
`û(x) = argmin_u J_θ(u) + ½‖u−x‖² ≈ u_PM(x)`. This convex program (ICNN + quadratic),
solved by L-BFGS. Ground truth `u_PM` is the sampler's. Lead
metric: **relative L2** (comparable to the prox residual above); PSNR and SSIM
alongside.


In [ ]:
res = denoise.run(image=IMAGE, arch=ARCH, sweeps=PM_SWEEPS,
                  sigma=SIGMA, t=T, ckpt=ckpt_path, steps=FIT_STEPS)
display(Image(filename=os.path.join(
    figures.OUT, f"denoise_{ARCH}_{IMAGE.split('_')[0]}{TAG}.png")))


## 5. Cost — the learned prior is cheaper than the sampler

The payoff of learning the prior, quantified. Computing `u_PM` needs MCMC **for
every new image**; the learned prior denoises with a **single sub-second convex
solve**, trained once and then amortized. Section 4 runs *both* only to compare
against ground truth — in deployment you run just the prox. (Times are for this
machine and one 256×256 image; observations.)


In [ ]:
import time
from scipy.io import loadmat
from tvpm.paths import IMAGES
from tvpm.denoise import tile, prox_of_Jtheta

img = np.asarray(loadmat(os.path.join(IMAGES, IMAGE + ".mat"))[IMAGE], float)
xn = np.clip(tile(img) + np.random.default_rng(3).normal(0, SIGMA, tile(img).shape), 0, 1)

from tvpm.sampler import sample_pm
t0 = time.time(); sample_pm(xn, SIGMA, LAM, sweeps=PM_SWEEPS, w=1.0, seed=100)
t_pm = time.time() - t0

mu_t, s_t = torch.tensor(units.mu).float(), torch.tensor(units.s).float()
t0 = time.time(); prox_of_Jtheta(model, mu_t, s_t, xn)
t_hat = time.time() - t0

print(f"u_PM   (sampler, m={PM_SWEEPS})            : {t_pm:7.1f} s   <- MCMC, rerun per image")
print(f"u_hat  (prox of the learned {ARCH} prior)  : {t_hat:7.2f} s   <- trained once, then this")
print(f"speedup                                    : {t_pm/t_hat:7.0f}x")
